# Cartesian Series Extrapolation

Minimal time-series notebook using `nf2.run_series`. Run a single extrapolation first and use it as `INITIAL_NF2_PATH`.

In [ ]:
from pathlib import Path

RUN_DOWNLOAD = False
RUN_TRAINING = False

JSOC_EMAIL = "you@example.org"
SHARP_NUM = 377
NOAA_NUM = None
T_START = "2011-02-15T00:00:00"
T_END = "2011-02-15T01:00:00"
CADENCE = "720s"
SERIES = "sharp_cea_720s"
SEGMENTS = "Br,Bp,Bt,Br_err,Bp_err,Bt_err"

RUN_DIR = Path("examples/runs/notebooks/cartesian_series")
DATA_DIR = RUN_DIR / "data"
WORK_DIR = RUN_DIR / "work"
INITIAL_NF2_PATH = Path("examples/runs/notebooks/sharp_cea/extrapolation_result.nf2")
NF2_PATH = None
EXPORT_DIR = RUN_DIR / "exports"

EXPORT_MM_PER_PIXEL = 1.44
HEIGHT_MIN = 0
HEIGHT_MAX = 80

In [ ]:
from pathlib import Path
from dateutil.parser import parse

import matplotlib.pyplot as plt
import numpy as np

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm
from nf2.evaluation.output_metrics import current_density

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DOWNLOAD:
    nf2.download_sharp_series(
        str(DATA_DIR), JSOC_EMAIL, parse(T_START), parse(T_END),
        noaa_num=NOAA_NUM, sharp_num=SHARP_NUM, cadence=CADENCE,
        segments=SEGMENTS, series=SERIES,
    )
print("Br files:", len(list(DATA_DIR.glob("*Br.fits"))))

In [ ]:
config = {
    "path": str(RUN_DIR),
    "work_path": str(WORK_DIR),
    "meta_path": str(INITIAL_NF2_PATH),
    "logging": {"project": "nf2", "name": "SHARP CEA series"},
    "data": {
        "geometry": "cartesian",
        "boundaries": [{"type": "fits", "data_path": str(DATA_DIR)}],
        "validation": [{"id": "cube", "type": "cube"}, {"id": "slices", "type": "slices"}],
        "z_range": [HEIGHT_MIN, HEIGHT_MAX],
    },
    "training": {"reload_dataloaders_every_n_epochs": 1},
}

if RUN_TRAINING:
    nf2.run_series(**config)
else:
    config

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_series([str(RUN_DIR / "*.nf2")], str(EXPORT_DIR), fmt="hdf5", overwrite=True, Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy"])

nf2_files = sorted(RUN_DIR.glob("*.nf2"))
NF2_PATH = Path(NF2_PATH) if NF2_PATH else (nf2_files[0] if nf2_files else None)
if NF2_PATH is None:
    raise FileNotFoundError("No .nf2 series state found yet.")

In [ ]:
out = nf2.load(NF2_PATH)
cube = out.load_cube(
    Mm_per_pixel=EXPORT_MM_PER_PIXEL,
    height_range=[HEIGHT_MIN, HEIGHT_MAX],
    metrics=["j", "free_energy"],
    progress=True,
)

def values(x):
    return np.asarray(getattr(x, "value", x))

b = values(cube["b"])
j = values(cube["metrics"]["j"])
free_energy = values(cube["metrics"]["free_energy"])
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j)),
    "sigma_J_1e2": float(sigma_J(b, j) * 1e2),
    "mean_B_norm": float(np.nanmean(vector_norm(b))),
    "mean_J_norm": float(np.nanmean(vector_norm(j))),
    "total_free_energy_density": float(np.nansum(free_energy)),
}
metrics

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

current_map = np.nansum(vector_norm(j), axis=2)
free_energy_map = np.nansum(free_energy, axis=2)
boundary_bz = b[:, :, 0, 2]

im = axs[0].imshow(current_map.T, origin="lower", cmap="magma")
axs[0].set_title("Integrated |J|")
plt.colorbar(im, ax=axs[0], fraction=0.046)

im = axs[1].imshow(free_energy_map.T, origin="lower", cmap="inferno")
axs[1].set_title("Free energy")
plt.colorbar(im, ax=axs[1], fraction=0.046)

lim = np.nanpercentile(np.abs(boundary_bz), 99)
im = axs[2].imshow(boundary_bz.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
axs[2].set_title("Model boundary Bz")
plt.colorbar(im, ax=axs[2], fraction=0.046)

for ax in axs:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()